In [11]:
import gradio as gr
import numpy as np
from PIL import Image

In [34]:
MAX_STYLES = 5 
IMG_HEIGHT = 300
class StyleBlock :
    def __init__(self, visible = False):
        with gr.Group(visible=visible) as block:
            self.img = gr.Image(interactive=True, height=300)
            self.weight = gr.Slider(label="Weight", minimum=0, maximum=1, value=1, interactive=True)
            self.scale = gr.Slider(label="Spatial scaling", minimum=0.1, maximum=3, value=1, interactive=True)
            self.rmv_btn = gr.Button(value="Remove style", visible=False)
        self.block = block
        self.img.input(self.resize, inputs=self.img, outputs=self.img) # Image is always resized to a fixed height to make sure all style blocks are correctly aligned

    def resize(self, img) :
        self.true_res_img = img # Keep the original image as it will be used for computation
        if img is None :
            return gr.update(value=None)
        img = Image.fromarray(img)
        print(img.height, img.width)
        new_width = int((IMG_HEIGHT / img.height) * img.width)
        new_img = img.resize((new_width, IMG_HEIGHT))
        return gr.update(value=new_img)

    def list(self) :
        return [self.img, self.weight, self.scale]
    
    def reset(self) :
        return (gr.update(value=None), gr.update(value=1), gr.update(value=1))
    
    def copy(self, new_img, new_weight, new_scale) :
        return (gr.update(value = new_img), gr.update(value = new_weight), gr.update(value = new_scale))


def params_block() :
    with gr.Blocks() as block :
        with gr.Row(equal_height=True) :
            with gr.Column(scale=2) :
                pres_col = gr.Checkbox(label="Preserve colors", value=False)
                # sep = gr.Markdown("---")
                with gr.Group() :
                    res_size = gr.Slider(label="Resize size", minimum=256, maximum=2000, value=1000, interactive=True)
                    keep_ratio = gr.Checkbox(label="Keep aspect ratio", value=False)
            with gr.Column(scale=1) :
                patches = gr.Checkbox(label="Work with patches", value=False)
                patch_size = gr.Slider(label="Patch size", minimum=256, maximum=1000, interactive=False)
                patch_context_size = gr.Slider(label="Patch context size", minimum=256, maximum=1500, interactive=False)
                patch_overlap = gr.Slider(label="Patch overlap", minimum=0, maximum=0.9, value=0.5, interactive=False)
            
            def update_patches_params(enable : bool) :
                return gr.update(interactive=enable), gr.update(interactive=enable), gr.update(interactive=enable)
            
            patches.change(update_patches_params, inputs=patches, outputs=[patch_size, patch_context_size, patch_overlap])
        return block




def generation_config_block(scale) :
    with gr.Blocks() as block :
        index_state = gr.State(1)
        style_blocks = []
        add_btn = gr.Button(value="➕ Add style", render=False)    
        def link_rmv_btns() :
            for i, block in enumerate(style_blocks) :
                block.rmv_btn.click(hide_block, inputs=[gr.State(i), index_state]  + [b.img for b in style_blocks] + [b.weight for b in style_blocks] + [b.scale for b in style_blocks], outputs = [b.block for b in style_blocks] + [add_btn] + [b.rmv_btn for b in style_blocks] + [item for b in style_blocks for item in b.list()] + [index_state])

        def hide_block(index, index_state, *blocks_values) :
            new_index = index_state - 1
            img_values = blocks_values[:MAX_STYLES]
            weight_values = blocks_values[MAX_STYLES:2*MAX_STYLES]
            scale_values = blocks_values[2*MAX_STYLES:3*MAX_STYLES]
            visible_updates = [gr.update(visible=True) if i < new_index else gr.update(visible=False) for i in range(MAX_STYLES)]
            add_btn_update = gr.update(visible=(new_index < MAX_STYLES))
            rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)] if new_index == 1 else [gr.update(visible=True) for _ in range(MAX_STYLES)]
            copy_updates = [[gr.update() for _ in range(3)] for _ in range(MAX_STYLES)]
            for i in range(index, MAX_STYLES - 1) :
                copy_updates[i] = style_blocks[i].copy(img_values[i+1], weight_values[i+1], scale_values[i+1])
            copy_updates[-1] = style_blocks[-1].reset()
            return (*visible_updates, add_btn_update, *rmv_btns_updates, *[item for cp_update in copy_updates for item in cp_update], new_index)

        def show_next_block(index) :
            new_index = index + 1
            updates = [gr.update(visible = (i <= index)) for i in range(MAX_STYLES)]
            button_update = gr.update(visible = (new_index < MAX_STYLES))
            if new_index == 1 :
                rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)]
            else : 
                rmv_btns_updates = [gr.update(visible=True) for _ in range(MAX_STYLES)]
            return (*updates, *rmv_btns_updates, new_index, button_update)
        

        with gr.Column(scale=scale) :
            content_part = gr.Image()
            with gr.Group() :
                with gr.Row(equal_height=True) as row :
                    for i in range(MAX_STYLES):
                        style_blocks.append(StyleBlock(visible = (i == 0)))
                    link_rmv_btns()
                    add_btn.render() 
                    
                    add_btn.click(show_next_block, inputs=index_state, outputs=[b.block for b in style_blocks] + [b.rmv_btn for b in style_blocks] + [index_state, add_btn])
                    
            params = params_block()
    return block

with gr.Blocks() as interface :
    with gr.Row(equal_height=True) :
        config = generation_config_block(scale=6)
        # with gr.Column(scale=1, min_width=10) :
        #     gr.HTML("""
        #                 <div style="display: flex; justify-content: center; align-items: stretch; height: 100%; background: red">
        #                     <div style="width: 1px; background-color: #ccc; flex-grow: 1;"></div>
        #                 </div>
        #             """)
        with gr.Column(scale=4) : 
            generated = gr.Image()

interface.launch()

* Running on local URL:  http://127.0.0.1:7887

To create a public link, set `share=True` in `launch()`.


775 1677
348 483
4000 3000
726 1352
440 538
1138 1171
334 608


In [ ]:
MAX_STYLES = 5 
IMG_HEIGHT = 300
class StyleBlock :
    def __init__(self, visible = False):
        with gr.Group(visible=visible) as block:
            self.img = gr.Image(interactive=True, height=300)
            self.weight = gr.Slider(label="Weight", minimum=0, maximum=1, value=1, interactive=True)
            self.scale = gr.Slider(label="Spatial scaling", minimum=0.1, maximum=3, value=1, interactive=True)
            self.rmv_btn = gr.Button(value="Remove style", visible=False)
        self.block = block
        self.img.input(self.resize, inputs=self.img, outputs=self.img) # Image is always resized to a fixed height to make sure all style blocks are correctly aligned

    def resize(self, img) :
        self.true_res_img = img # Keep the original image as it will be used for computation
        if img is None :
            return gr.update(value=None)
        img = Image.fromarray(img)
        print(img.height, img.width)
        new_width = int((IMG_HEIGHT / img.height) * img.width)
        new_img = img.resize((new_width, IMG_HEIGHT))
        return gr.update(value=new_img)

    def list(self) :
        return [self.img, self.weight, self.scale]
    
    def reset(self) :
        return (gr.update(value=None), gr.update(value=1), gr.update(value=1))
    
    def copy(self, new_img, new_weight, new_scale) :
        return (gr.update(value = new_img), gr.update(value = new_weight), gr.update(value = new_scale))

with gr.Blocks() as demo :
    style_blocks = []
    index_state = gr.State(1)
    add_btn = gr.Button(value="➕ Add style", render=False)

    def link_rmv_btns() :
        for i, block in enumerate(style_blocks) :
           block.rmv_btn.click(hide_block, inputs=[gr.State(i), index_state]  + [b.img for b in style_blocks] + [b.weight for b in style_blocks] + [b.scale for b in style_blocks], outputs = [b.block for b in style_blocks] + [add_btn] + [b.rmv_btn for b in style_blocks] + [item for b in style_blocks for item in b.list()] + [index_state])

    def hide_block(index, index_state, *blocks_values) :
        new_index = index_state - 1
        img_values = blocks_values[:MAX_STYLES]
        weight_values = blocks_values[MAX_STYLES:2*MAX_STYLES]
        scale_values = blocks_values[2*MAX_STYLES:3*MAX_STYLES]
        visible_updates = [gr.update(visible=True) if i < new_index else gr.update(visible=False) for i in range(MAX_STYLES)]
        add_btn_update = gr.update(visible=(new_index < MAX_STYLES))
        rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)] if new_index == 1 else [gr.update(visible=True) for _ in range(MAX_STYLES)]
        copy_updates = [[gr.update() for _ in range(3)] for _ in range(MAX_STYLES)]
        for i in range(index, MAX_STYLES - 1) :
            copy_updates[i] = style_blocks[i].copy(img_values[i+1], weight_values[i+1], scale_values[i+1])
        copy_updates[-1] = style_blocks[-1].reset()
        return (*visible_updates, add_btn_update, *rmv_btns_updates, *[item for cp_update in copy_updates for item in cp_update], new_index)

    with gr.Row(equal_height=True) as row :
        for i in range(MAX_STYLES):
            style_blocks.append(StyleBlock(visible = (i == 0)))
        link_rmv_btns()
        add_btn.render()

    @add_btn.click(inputs=index_state, outputs=[b.block for b in style_blocks] + [b.rmv_btn for b in style_blocks] + [index_state, add_btn])
    def show_next_block(index) :
        new_index = index + 1
        updates = [gr.update(visible = (i <= index)) for i in range(MAX_STYLES)]
        button_update = gr.update(visible = (new_index < MAX_STYLES))
        if new_index == 1 :
            rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)]
        else : 
            rmv_btns_updates = [gr.update(visible=True) for _ in range(MAX_STYLES)]
        return (*updates, *rmv_btns_updates, new_index, button_update)

demo.launch()

* Running on local URL:  http://127.0.0.1:7880

To create a public link, set `share=True` in `launch()`.


4000 3000
348 483
770 1723
413 186
757 2071
